# 07 · CD con los parámetros del compensador (CDC)

**Objetivo.** Probar el emulador de dispersión cromática con el punto de diseño que usa el grupo del compensador (CDC por overlap-save en frecuencia), para:

- verificar que el bloque `ChromaticDispersion` reproduce la CD ideal con esos parámetros (60 GBd, 72 GSa/s, 200 km);
- comprobar que un compensador con su ventana (FFT de 384 muestras, avance de 192) recupera la señal, con y sin ruido;
- dejar explícita la **convención de signo** que tiene que usar el CDC ($H^*$, decisión 2) y el **alcance** de su ventana.

Parámetros tomados de la presentación del grupo del CDC ("Parámetros del sistema simulado").

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import resample_poly, welch

from emulador import plots
from emulador.blocks import ChromaticDispersion
from emulador.cd import cd_N, cd_R, cd_hbar, cd_overlap_save, cd_reference, cd_summary
from emulador.config import D_si
from emulador.metrics import evm_opt, nrmse
from emulador.stimuli import map_gray, prbs

plots.use_style()

## 1. Parámetros del grupo del CDC

| Bloque | Parámetro | Valor |
|---|---|---|
| Señal | Tasa de símbolos | 60 GBd |
| | Frecuencia de muestreo | 72 GSa/s (sobremuestreo 1,2) |
| | Modulación | 16-QAM Gray |
| | Largo | 10 000 símbolos |
| | Conformador | coseno realzado, roll-off 0,1, 300 símbolos |
| Canal | Fibra | 200 km, D = 17 ps/(nm·km), λ = 1550 nm |
| | Ruido | OSNR 20 dB |
| CDC | Ventana FFT (B) | 384 muestras |
| | Avance por ciclo (H) | 192 muestras (50 % de solapamiento) |
| | Taps del ecualizador (F) | 256 (fuera del alcance de esta prueba) |

**Equivalencia con el emulador.** El overlap-save del emulador usa una FFT de $L = 2N$ con $N$ muestras nuevas por bloque: la ventana del CDC (B = 384, H = 192) es exactamente $L = 384$, $N = 192$. El emulador acepta ese $N$ explícito (tiene que ser par y $N \ge R$).

**Diferencias con el escenario del emulador.**
- 1,2 muestras por símbolo: el `Scenario` del emulador solo admite 1, 2, 4, 8 o 16 sps, pero el bloque de CD recibe $f_s$ directamente y no depende de eso. La señal se genera acá (sección 3).
- Coseno realzado (RC) completo en el transmisor, no raíz (RRC): la señal ya cumple Nyquist y el receptor muestrea sin filtro adaptado.
- N = 192 no es potencia de 2; `cd_N` elegiría 256 para el hardware del emulador (decisión 5). Para el CDC es una elección de su diseño.
- Sin β₃ (S = 0), como en su simulación.

## 2. Parámetros y magnitudes derivadas

In [ ]:
RS = 60e9
FS = 72e9
OS = FS / RS                       # 1,2
ROLLOFF, SPAN = 0.1, 300           # RC del transmisor
N_SYM = 10_000
D_PS, LAM_NM, Z_KM = 17.0, 1550.0, 200.0
OSNR_DB = 20.0
N_CDC = 192                        # avance H; ventana B = 2·N = 384

D, lam, z = D_si(D_PS), LAM_NM / 1e9, Z_KM * 1e3
R = cd_R(FS, D, lam, z)


class _Esc:  # cd_summary solo usa fs, Rs y wavelength
    fs, Rs, wavelength = FS, RS, lam


s = cd_summary(_Esc, D, z)
print(f"sobremuestreo = {OS:.1f} muestras/símbolo")
print(f"R = {R} muestras  (chirp de la CD a {FS / 1e9:.0f} GSa/s y {Z_KM:.0f} km)")
print(f"N del CDC = {N_CDC} (ventana {2 * N_CDC}); N >= R: {N_CDC >= R}; margen = {N_CDC - R} muestras")
print(f"cd_N (potencia de 2 del emulador) = {cd_N(FS, D, lam, z)}")
print(f"retardo del overlap-save = N/2 = {N_CDC // 2} muestras = {N_CDC / 2 / FS * 1e12:.0f} ps")
print(f"dispersión acumulada = {s['Dz_ps_nm']:.0f} ps/nm; pendiente del retardo de grupo = {s['gd_slope_ps_GHz']:.1f} ps/GHz")
print(f"ensanchamiento en el borde de la señal (±{(1 + ROLLOFF) * RS / 2e9:.0f} GHz): "
      f"±{abs(s['gd_slope_ps_GHz']) * (1 + ROLLOFF) * RS / 2e9:.0f} ps = ±{abs(s['gd_slope_ps_GHz']) * (1 + ROLLOFF) * RS / 2e9 * 1e-12 * RS:.1f} símbolos")

## 3. Señal: 16-QAM Gray con coseno realzado a 1,2 sps

Los símbolos salen de la PRBS-15 del emulador con mapeo Gray (potencia media 1; en la presentación del CDC los niveles no están normalizados, ±1 y ±3, lo que solo cambia la escala). Se conforman con un RC de roll-off 0,1 y 300 símbolos a 6 sps y se toma una de cada 5 muestras: 6/5 = 1,2 sps. No hay aliasing porque la señal ocupa $\pm(1+\rho)R_s/2$ = ±33 GHz y Nyquist está en ±36 GHz.

El receptor de esta notebook interpola por 5 (vuelve a 6 sps) y toma una muestra por símbolo; la EVM se mide con ganancia compleja óptima (`metrics.evm_opt`), descartando 200 símbolos en cada borde.

In [ ]:
UP, DOWN = 6, 5          # 6 sps -> /5 -> 1,2 sps
DISCARD = 200            # símbolos descartados por borde en la EVM


def rc_taps(beta, span, sps):
    # Coseno realzado (no raíz): h(t) = sinc(t)·cos(πβt) / (1 - (2βt)²), t en símbolos
    t = np.arange(-span * sps // 2, span * sps // 2 + 1) / sps
    h = np.sinc(t) * np.cos(np.pi * beta * t)
    den = 1 - (2 * beta * t) ** 2
    sing = np.isclose(den, 0)
    h[~sing] /= den[~sing]
    h[sing] = np.pi / 4 * np.sinc(1 / (2 * beta))
    return h


def rx_symbols(y):
    # 1,2 sps -> 6 sps (interpolación polifásica) y una muestra por símbolo
    return resample_poly(y, DOWN, 1)[: N_SYM * UP][::UP]


sym = map_gray(prbs(15, 4 * N_SYM), "qam16")
up = np.zeros(N_SYM * UP, complex)
up[::UP] = sym
x6 = np.convolve(up, rc_taps(ROLLOFF, SPAN, UP), mode="same")
x = x6[::DOWN]
x = x / np.sqrt(np.mean(np.abs(x) ** 2))
evm_b2b = evm_opt(rx_symbols(x), sym, DISCARD)

f, p = welch(x, fs=FS, nperseg=2048, return_onesided=False)
o = np.argsort(f)
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(f[o] / 1e9, 10 * np.log10(p[o] / p.max()))
for sgn in (-1, 1):
    ax.axvline(sgn * (1 + ROLLOFF) * RS / 2e9, ls="--", color="C1", lw=1)
    ax.axvline(sgn * FS / 2e9, ls=":", color="k", lw=1)
ax.set_ylim(-80, 3)
ax.set_xlabel("f [GHz]")
ax.set_ylabel("PSD [dB]")
ax.set_title("16-QAM, 60 GBd, RC roll-off 0,1, 72 GSa/s (-- : ±33 GHz; ··· : ±fs/2)")
plots.save_fig(fig, "07_espectro")
print(f"{x.size} muestras; EVM back-to-back = {100 * evm_b2b:.2f} %")

**Qué muestra la figura.** La señal ocupa ±33 GHz (líneas de trazos) y no llega a ±36 GHz (Nyquist a 72 GSa/s, punteadas): el sobremuestreo de 1,2 alcanza sin aliasing, con solo 3 GHz de margen a cada lado. Sin canal, la EVM es el piso de esta cadena (RC truncado e interpolación del receptor).

## 4. CD de 200 km con el emulador

Se aplica el bloque `ChromaticDispersion` con N = 192 (la ventana del CDC) y con N = 256 (`cd_N`), y se compara con la referencia ideal (`cd_reference`, una FFT de toda la señal). Se descartan N muestras por borde.

In [ ]:
y_cd, err_cd = {}, {}
ref = cd_reference(x, FS, D, lam, z)
for N in (N_CDC, cd_N(FS, D, lam, z)):
    blk = ChromaticDispersion(fs=FS, D_ps_nm_km=D_PS, wavelength_nm=LAM_NM, length_km=Z_KM, N=N)
    y_cd[N] = blk(x)
    err_cd[N] = nrmse(y_cd[N][N:-N], ref[N:-N])
    print(f"N = {N}: NRMSE contra la referencia = {err_cd[N]:.1e}")
evm_cd = evm_opt(rx_symbols(y_cd[N_CDC]), sym, DISCARD)
print(f"EVM después de la CD (sin compensar) = {100 * evm_cd:.1f} %")

## 5. Compensación con la ventana del CDC

El compensador se emula con el mismo overlap-save (B = 384, H = 192) y el filtro conjugado, $H^*$, que es la CD con $-D$. Es un CDC ideal en punto flotante: sirve de referencia para el diseño del grupo. Se prueba:

- sin ruido, para ver el error propio de la ventana;
- con ruido de OSNR 20 dB, agregado entre la fibra y el CDC;
- con el **signo equivocado** ($H$ en lugar de $H^*$), el error que hay que evitar.

**Ruido.** El bloque AWGN del emulador todavía no está en esta rama (lo implementa otra tesis), así que se agrega acá. La OSNR se mide en 12,5 GHz; para una polarización, la relación con la SNR por símbolo es $E_s/N_0 = \mathrm{OSNR}\cdot 2B_{ref}/R_s$: OSNR 20 dB equivale a $E_s/N_0$ ≈ 16,2 dB a 60 GBd. (Si en la simulación del CDC se usa el valor de 20 dB directamente como SNR, ese caso también se muestra.)

In [ ]:
B_REF = 12.5e9


def add_noise(y, esn0_db, seed=1):
    # ruido complejo blanco en toda la banda fs; Es/N0 referido a la tasa de símbolos
    rng = np.random.default_rng(seed)
    ps = np.mean(np.abs(y) ** 2)
    sigma2 = ps * (FS / RS) / 10 ** (esn0_db / 10)
    return y + np.sqrt(sigma2 / 2) * (rng.standard_normal(y.size) + 1j * rng.standard_normal(y.size))


esn0_osnr = OSNR_DB + 10 * np.log10(2 * B_REF / RS)
cdc = ChromaticDispersion(fs=FS, D_ps_nm_km=-D_PS, wavelength_nm=LAM_NM, length_km=Z_KM, N=N_CDC)   # H*
cdc_wrong = ChromaticDispersion(fs=FS, D_ps_nm_km=D_PS, wavelength_nm=LAM_NM, length_km=Z_KM, N=N_CDC)  # H
y = y_cd[N_CDC]
cases = {
    "Sin canal": x,
    "CD 200 km": y,
    "CD + CDC (H*)": cdc(y),
    f"CD + OSNR 20 dB (Es/N0 {esn0_osnr:.1f} dB) + CDC": cdc(add_noise(y, esn0_osnr)),
    "CD + SNR 20 dB + CDC": cdc(add_noise(y, OSNR_DB)),
    "CD + CDC con signo equivocado (H)": cdc_wrong(y),
}
evm = {k: evm_opt(rx_symbols(v), sym, DISCARD) for k, v in cases.items()}

show = ["Sin canal", "CD 200 km", "CD + CDC (H*)", f"CD + OSNR 20 dB (Es/N0 {esn0_osnr:.1f} dB) + CDC"]
fig, axs = plt.subplots(1, 4, figsize=(18, 4.6))
for ax, k in zip(axs, show):
    s_rx = rx_symbols(cases[k])[DISCARD:-DISCARD]
    ref_s = sym[DISCARD:-DISCARD]
    g = np.vdot(s_rx, ref_s) / np.vdot(s_rx, s_rx)        # ganancia compleja óptima, como evm_opt
    plots.constellation(ax, g * s_rx, ref=ref_s, title=f"{k.split(' (Es')[0]} · EVM {100 * evm[k]:.1f} %")
plots.save_fig(fig, "07_constelaciones")

for k, e in evm.items():
    print(f"{k:48s} EVM = {100 * e:6.2f} %   ({-20 * np.log10(e):5.1f} dB)")

**Qué muestra la figura.** Sin canal, la constelación es limpia. Después de 200 km de CD es una nube sin estructura (el retardo de grupo llega a ±0,9 ns en el borde de la banda: cada símbolo recibido mezcla unos ±54 símbolos vecinos). El CDC con la ventana de 384 y el filtro conjugado la recupera casi por completo: el error que queda es el de la ventana finita. Con ruido de OSNR 20 dB, la EVM la domina el ruido, no la compensación.

## 6. Alcance de la ventana del CDC

¿Hasta qué distancia alcanza la ventana de 384 (N = 192)? Se barre z y se mide el NRMSE del emulador contra la referencia y la EVM después de compensar, sin ruido. La condición de diseño es $N \ge R$.

In [ ]:
Z_SWEEP = np.arange(25, 401, 25)
nrmse_z, evm_z, R_z = [], [], []
for zk in Z_SWEEP:
    zz = zk * 1e3
    R_z.append(cd_R(FS, D, lam, zz))
    r_ = cd_reference(x, FS, D, lam, zz)
    m = N_CDC
    # emulador con N = 192 (funciones, porque el bloque exige N >= R) contra la referencia
    ols = cd_overlap_save(x, cd_hbar(N_CDC, FS, D, lam, zz))[N_CDC // 2 : N_CDC // 2 + x.size]
    nrmse_z.append(nrmse(ols[m:-m], r_[m:-m]))
    # CDC con la ventana de 384 sobre el canal ideal, para aislar el error del CDC
    comp = cd_overlap_save(r_, cd_hbar(N_CDC, FS, -D, lam, zz))[N_CDC // 2 : N_CDC // 2 + x.size]
    evm_z.append(evm_opt(rx_symbols(comp), sym, DISCARD))
z_max = max(zk for zk in range(1, 1000) if cd_R(FS, D, lam, zk * 1e3) <= N_CDC)  # último km con R <= N

fig, axs = plt.subplots(1, 2, figsize=(13, 4.3))
axs[0].semilogy(Z_SWEEP, nrmse_z, "o-")
axs[0].axhline(1e-2, ls=":", color="k")
axs[0].set_title("Emulador (N = 192) contra la referencia: NRMSE")
axs[1].semilogy(Z_SWEEP, 100 * np.array(evm_z), "o-", color="C1")
axs[1].axhline(100 * evm_b2b, ls=":", color="k", label="piso sin canal")
axs[1].set_title("CDC (ventana 384) sobre la CD ideal: EVM [%]")
axs[1].legend()
for ax in axs:
    ax.axvline(z_max, ls="--", color="C2")
    ax.set_xlabel("z [km]")
plots.save_fig(fig, "07_alcance_ventana")

print(f"N = {N_CDC} cubre R hasta z = {z_max} km (R({z_max} km) = {cd_R(FS, D, lam, z_max * 1e3)}, "
      f"R({z_max + 1} km) = {cd_R(FS, D, lam, (z_max + 1) * 1e3)})")
for zk, r_, e1, e2 in zip(Z_SWEEP, R_z, nrmse_z, evm_z):
    print(f"{zk:4d} km  R = {r_:4d}  NRMSE = {e1:.1e}  EVM con CDC = {100 * e2:6.2f} %")

**Qué muestra la figura.** Mientras $R \le 192$, es decir hasta 271 km (línea de trazos), tanto el emulador como el CDC con la ventana de 384 tienen error chico; cuando el chirp supera la ventana, el error sube de golpe, igual que en la Fig. 9 del notebook 04. Los 200 km del escenario quedan dentro del alcance, con margen.

## 7. Validación

In [ ]:
assert R == 141 and N_CDC >= R
assert evm_b2b < 0.005
assert err_cd[N_CDC] < 1e-2 and err_cd[256] < err_cd[N_CDC]
assert evm_cd > 0.5
assert evm["CD + CDC (H*)"] < 0.01
assert evm["CD + CDC con signo equivocado (H)"] > 0.5
assert evm[f"CD + OSNR 20 dB (Es/N0 {esn0_osnr:.1f} dB) + CDC"] > evm["CD + CDC (H*)"]
assert z_max == 271
print("OK")

## 8. Conclusiones

- **El emulador funciona con el punto de diseño del CDC.** A 72 GSa/s y 200 km el chirp ocupa R = 141 muestras; la ventana del CDC (B = 384, H = 192) equivale al overlap-save del emulador con N = 192 ≥ R. Con ese N el bloque `ChromaticDispersion` sigue a la CD ideal con NRMSE 3,1e-3 (1,7e-3 con N = 256, el que elegiría `cd_N`).
- **Sin compensar, 200 km destruyen la señal**: EVM 99,5 %. El retardo de grupo llega a ±0,9 ns en el borde de la banda, unos ±54 símbolos.
- **Un CDC ideal con su ventana recupera la constelación**: EVM 0,32 % (piso de esta cadena sin canal: 0,15 %). El error propio de la ventana es chico frente a cualquier ruido realista.
- **Con OSNR 20 dB** (Es/N0 ≈ 16,2 dB para una polarización a 60 GBd) la EVM después del CDC es 16,1 %, dominada por el ruido. Si el valor de 20 dB se usa como SNR directamente, la EVM es 10,5 %: conviene acordar con el grupo cuál de las dos definiciones usan.
- **Signo**: compensar con $H$ en lugar de $H^*$ duplica la dispersión (EVM 99,7 %). Con la convención positiva del emulador (decisión 2), el CDC tiene que usar el conjugado.
- **Alcance de la ventana**: N = 192 cubre R hasta **271 km** a 72 GSa/s. Más allá, el error sube de golpe (EVM del CDC 0,8 % a 275 km, 1,9 % a 300 km, 13 % a 400 km). Para distancias mayores, la ventana tiene que crecer con R ∝ z·f_s².
- **Para el grupo del CDC**: el emulador puede generar sus vectores de prueba con estos parámetros (señal a 1,2 sps y bloque de CD con `fs = 72e9`); la salida cruda `process_hw` sale retrasada N/2 = 96 muestras si se usa N = 192.

Figuras en `docs/golden_model/figuras/07_*.png`.